In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('datasets')

finance_df = pd.read_json(DATA_DIR / 'finance.jsonl', lines=True)
medicine_df = pd.read_json(DATA_DIR / 'medicine.jsonl', lines=True)
open_qa_df = pd.read_json(DATA_DIR / 'open_qa.jsonl', lines=True)
reddit_eli5_df = pd.read_json(DATA_DIR / 'reddit_eli5.jsonl', lines=True)
wiki_csai_df = pd.read_json(DATA_DIR / 'wiki_csai.jsonl', lines=True)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

def df_to_text_label(df, source_name):
    human = (
        df[['human_answers']]
        .explode('human_answers')
        .dropna()
        .rename(columns={'human_answers': 'text'})
    )
    human['label'] = 0

    ai = (
        df[['chatgpt_answers']]
        .explode('chatgpt_answers')
        .dropna()
        .rename(columns={'chatgpt_answers': 'text'})
    )
    ai['label'] = 1

    human['source'] = source_name
    ai['source'] = source_name

    return pd.concat([human, ai], ignore_index=True)

datasets = {
    "finance": df_to_text_label(finance_df, "finance"),
    "medicine": df_to_text_label(medicine_df, "medicine"),
    "open_qa": df_to_text_label(open_qa_df, "open_qa"),
    "reddit_eli5": df_to_text_label(reddit_eli5_df, "reddit_eli5"),
    "wiki_csai": df_to_text_label(wiki_csai_df, "wiki_csai"),
}

for name, df in datasets.items():
    print(name, df.shape)

finance (8436, 3)
medicine (2585, 3)
open_qa (4748, 3)
reddit_eli5 (67996, 3)
wiki_csai (1684, 3)


In [3]:
results = []

for test_name, test_df in datasets.items():
    train_domains = [df for name, df in datasets.items() if name != test_name]
    train_df = pd.concat(train_domains, ignore_index=True)

    X_train, y_train = train_df['text'], train_df['label']
    X_test, y_test = test_df['text'], test_df['label']

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=200000,
            ngram_range=(1, 2),
            min_df=2
        )),
        ("logreg", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
        )),
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results.append(acc)

    print(f"=== Leave out {test_name} (train on the other 4) ===")
    print(f"Accuracy: {acc:.4f}")
    print()

avg_acc = sum(acc for acc in results) / len(results)
print("Average accuracy over all leave-one-domain-out runs:", avg_acc)

=== Leave out finance (train on the other 4) ===
Accuracy: 0.9339

=== Leave out medicine (train on the other 4) ===
Accuracy: 0.9760

=== Leave out open_qa (train on the other 4) ===
Accuracy: 0.6763

=== Leave out reddit_eli5 (train on the other 4) ===
Accuracy: 0.9067

=== Leave out wiki_csai (train on the other 4) ===
Accuracy: 0.7987

Average accuracy over all leave-one-domain-out runs: 0.8583098227480812


### Interpreting Results

From the above cell we can see that the logistic regression model trained is able to generalize across domains. However we should note that when the `open_qa` and `wiki_csai` datasets are left out of training, the accuracy score is observably worse. Additionally, the largest bulk of our dataset (`reddit_eli5`) being left out still resulted in a high accuracy score.

As such, we conclude that the most important datasets that contain features that can identify chatgpt generated text are the `open_qa` and `wiki_csai` datasets.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

full_df = pd.concat(datasets.values(), ignore_index=True)

for ngram_range in [(1,2), (1,3)]:
    temp_vec = TfidfVectorizer(
        min_df=2,
        ngram_range=ngram_range
    )
    temp_vec.fit(full_df['text'])
    print(f"ngram_range={ngram_range} = {len(temp_vec.vocabulary_):,} unique tokens")

ngram_range=(1, 2) = 770,772 unique tokens
ngram_range=(1, 3) = 2,029,460 unique tokens


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

# Combine all domains
full_df = pd.concat(datasets.values(), ignore_index=True)
X_all, y_all = full_df['text'], full_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("logreg", LogisticRegression(max_iter=1000, n_jobs=-1)),
])

param_grid = [
    {
        "tfidf__ngram_range":  [(1, 2)],
        "tfidf__max_features": [200000, 300000, 500000],
        "logreg__C":           [0.1, 1.0, 10.0],
    },
    {
        "tfidf__ngram_range":  [(1, 3)],
        "tfidf__max_features": [500000, 1000000, 1500000],
        "logreg__C":           [0.1, 1.0, 10.0],
    },
]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 2); total time=  25.9s


/Users/aikchong/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 2); total time=  26.0s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 2); total time=  26.2s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 2); total time=  26.2s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 2); total time=  27.2s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 2); total time=  27.2s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 2); total time=  27.0s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 2); total time=  27.3s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 2); total time=  25.5s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 2); total time=  25.4s
[CV] END logreg__C=0.1, tfidf__max_features=500000, tfidf__ngram_range=(1, 2); total time=  26.9s
[CV] END logreg__C=1

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('logreg',
                                        LogisticRegression(max_iter=1000,
                                                           n_jobs=-1))]),
             n_jobs=-1,
             param_grid=[{'logreg__C': [0.1, 1.0, 10.0],
                          'tfidf__max_features': [200000, 300000, 500000],
                          'tfidf__ngram_range': [(1, 2)]},
                         {'logreg__C': [0.1, 1.0, 10.0],
                          'tfidf__max_features': [500000, 1000000, 1500000],
                          'tfidf__ngram_range': [(1, 3)]}],
             scoring='f1', verbose=2)

In [6]:
print("Best params:", grid_search.best_params_)
print("Best CV F1: ", grid_search.best_score_)

final_model = grid_search.best_estimator_
y_pred = final_model.predict(X_test)
print("\nFinal Test Set Performance:")
print(classification_report(y_test, y_pred))

Best params: {'logreg__C': 10.0, 'tfidf__max_features': 500000, 'tfidf__ngram_range': (1, 3)}
Best CV F1:  0.9713499730456361

Final Test Set Performance:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     11709
           1       0.98      0.97      0.97      5381

    accuracy                           0.98     17090
   macro avg       0.98      0.98      0.98     17090
weighted avg       0.98      0.98      0.98     17090



In [7]:
amazon_df = pd.read_csv(DATA_DIR / 'amazon_review_dataset.csv')

amazon_df.head()

,name,main_category,sub_category,image,link,no_of_ratings,discount_price,actual_price,review_rating,original_review_text,...,t5_rouge1,t5_rouge2,t5_rougeL,t5_avg_rouge,best_model,raw_best_summary,best_avg_score,cleaned_narrative,qwen_used,processing_time_seconds
0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,965,"₹10,999","₹18,999",3.0 out of 5 stars,NOTE:,...,0.000000,0.000000,0.000000,0.000000,BART,WARNING: GRAPHIC CONTENT. CLICK HERE to read t...,0.000000,"""The customer mentioned that while they found ...",True,5.779507
1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,"113,956","₹18,999","₹19,999",1.0 out of 5 stars,Very bad experience with this device xr phone....,...,0.735294,0.727273,0.735294,0.732620,BART,Very bad experience with this device xr phone....,0.867236,The customer mentioned their extremely negativ...,True,4.791277
2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,"90,304","₹1,999","₹2,299",5.0 out of 5 stars,Amazing phone with amazing camera coming from ...,...,0.666667,0.650000,0.666667,0.661111,T5,amazon.com: amazing phone with amazing camera ...,0.661111,"""The customer was thoroughly impressed with th...",True,3.900565
3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,"24,863","₹15,999","₹24,999",1.0 out of 5 stars,So I got the device XR just today. The product...,...,0.564103,0.447368,0.512821,0.508097,T5,the face ID is not working and there's a glitc...,0.508097,The customer mentioned that the Face ID featur...,True,4.366925
4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,"113,956","₹18,999","₹19,999",5.0 out of 5 stars,I've been an android user all my life until I ...,...,0.677165,0.624000,0.661417,0.654194,T5,i've been an android user all my life until I ...,0.654194,The customer mentioned that they were previous...,True,4.871587


In [9]:
review_texts = amazon_df['original_review_text'].dropna()
review_predictions = final_model.predict(review_texts)

total_reviews = len(review_predictions)
classified_human = (review_predictions == 0).sum()
classified_ai = (review_predictions == 1).sum()

print(f"Total reviews:          {total_reviews}")
print(f"Classified as human:    {classified_human} ({classified_human/total_reviews*100:.1f}%)")
print(f"Classified as AI:       {classified_ai} ({classified_ai/total_reviews*100:.1f}%)")

Total reviews:          5007
Classified as human:    4985 (99.6%)
Classified as AI:       22 (0.4%)
